# 03 Validate the recovered features

Compares the lifetime set, which follows the construction of Elliptic++, with the features released by Elmougy and Liu
for every labelled address matched to the ledger, and checks the exact identities among the features.

1. Tight check: the share of addresses whose recomputed value matches the released one, per feature.
2. One-percent check: the addresses that diverge from the released values, under two absolute floors.
3. Fee share: the per-transaction definition against a single ratio of totals.
4. Identities among the features, on the training addresses.

Expected: all 55 features match for at least 99.9% of the 265,352 addresses, the least-matching one being the total
counterparty count at 99.915%, and 86 divergent addresses under the absolute floor of 1e-4 bitcoin.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

ROOT      = Path.cwd().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'outputs'
WALLETS   = RAW / 'elliptic_pp' / 'wallets_features_classes_combined.csv'
FEATURES_LIFETIME  = PROCESSED / 'features_lifetime.parquet'
FEATURES_RESISTANT = PROCESSED / 'features_leakage_resistant.parquet'
pd.set_option('display.width', 200)

STATS = ['total', 'min', 'max', 'mean', 'median']
FAMILIES = {
    'Volumes':      [f'btc_{q}_{s}' for q in ['transacted', 'sent', 'received'] for s in STATS],
    'Fees':         [f'fees_{s}' for s in STATS] + [f'fees_as_share_{s}' for s in STATS],
    'Block gaps':   [f'blocks_btwn_{q}_{s}' for q in ['txs', 'input_txs', 'output_txs'] for s in STATS],
    'Counterparty': [f'transacted_w_address_{s}' for s in STATS] + ['num_addr_transacted_multiple'],
    'Counts':       ['total_txs', 'num_txs_as_sender', 'num_txs_as receiver'],
    'Lifetime':     ['first_block_appeared_in', 'last_block_appeared_in', 'lifetime_in_blocks',
                     'first_sent_block', 'first_received_block', 'num_timesteps_appeared_in'],
}
FEATURES = [f for fs in FAMILIES.values() for f in fs]
assert len(FEATURES) == 55

con = duckdb.connect()
released = con.execute(f"""
    SELECT * FROM read_csv('{WALLETS}', header=true) WHERE "class" IN (1, 2)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY address ORDER BY "Time step") = 1
    ORDER BY address""").fetchdf()
lifetime  = pd.read_parquet(FEATURES_LIFETIME).sort_values('address').reset_index(drop=True)
resistant = pd.read_parquet(FEATURES_RESISTANT).sort_values('address').reset_index(drop=True)
print(f'released {len(released):,}  lifetime {len(lifetime):,}  leakage-resistant {len(resistant):,}')

BTC   = [f for f in FEATURES if f.startswith(('btc_', 'fees_'))]
OTHER = [f for f in FEATURES if f not in BTC]
MEANS = [f for f in OTHER if f.endswith('_mean')]
EXACT = [f for f in OTHER if f not in MEANS]

m = lifetime.merge(released, on='address', suffixes=('_c', '_p'))
C = {f: m[f'{f}_c'].to_numpy(float) for f in FEATURES}
P = {f: m[f'{f}_p'].to_numpy(float) for f in FEATURES}
print(f'addresses compared {len(m):,}')

released 265,354  lifetime 265,352  leakage-resistant 265,352
addresses compared 265,352


## 1. Tight check

A recomputed value $c$ matches a released value $p$ if $|c - p| \le 10^{-8} + 10^{-6}|p|$.
For the 26 integer-valued features the bound is below one, so the rule requires equality, and they are compared exactly.
The 4 means of integer quantities and the 25 amounts in bitcoin are compared with the tolerance.

In [2]:
def match(f, rtol=1e-6, atol=1e-8):
    if f in EXACT:
        return C[f] == P[f]
    return np.isclose(C[f], P[f], rtol=rtol, atol=atol)

tight = pd.Series({f: match(f).mean() for f in FEATURES}, name='match rate')
print(f'features matching for at least 99.9% of addresses: {(tight >= 0.999).sum()}/55')
print(f'least-matching feature: {tight.idxmin()} {tight.min():.5%}')
print()
print(tight.sort_values().to_frame().to_string(float_format=lambda v: f'{v:.5%}'))

features matching for at least 99.9% of addresses: 55/55
least-matching feature: transacted_w_address_total 99.91521%

                               match rate
transacted_w_address_total      99.91521%
transacted_w_address_mean       99.94385%
fees_as_share_mean              99.95063%
blocks_btwn_txs_mean            99.95101%
fees_as_share_total             99.95139%
blocks_btwn_txs_total           99.95591%
fees_as_share_median            99.95704%
num_addr_transacted_multiple    99.95892%
transacted_w_address_max        99.95930%
blocks_btwn_txs_max             99.96872%
fees_as_share_min               99.96872%
btc_transacted_total            99.96985%
transacted_w_address_median     99.97098%
btc_received_total              99.97324%
btc_transacted_mean             99.97362%
btc_received_mean               99.97400%
blocks_btwn_txs_median          99.97550%
btc_transacted_median           99.97852%
btc_received_max                99.98040%
btc_received_median             99.98078%

## 2. One-percent check

An address diverges if at least one of its 25 amounts differs from the released value by more than
$a + 0.01|p|$. The paper uses $a = 10^{-4}$ bitcoin (10,000 satoshis); the appendix compares it with $a = 10^{-8}$ (one satoshi).

In [3]:
FLOORS = {'1e-4': 1e-4, '1e-8': 1e-8}
divergent, per_feature = {}, {}
for name, a in FLOORS.items():
    fails = {f: ~np.isclose(C[f], P[f], rtol=1e-2, atol=a) for f in BTC}
    divergent[name] = np.logical_or.reduce(list(fails.values()))
    per_feature[name] = {f: int(v.sum()) for f, v in fails.items()}
    m.loc[divergent[name], ['address']].to_parquet(PROCESSED / f'divergent_{name}.parquet', index=False)
    print(f'floor {name}: {divergent[name].sum():,} divergent addresses ({divergent[name].mean():.4%})')
print()
pf = pd.DataFrame(per_feature)
print('divergent addresses per feature')
print(pf[pf.sum(axis=1) > 0].sort_values('1e-8', ascending=False).to_string())

floor 1e-4: 86 divergent addresses (0.0324%)
floor 1e-8: 137 divergent addresses (0.0516%)

divergent addresses per feature
                       1e-4  1e-8
fees_as_share_mean       25   115
fees_as_share_median     26   103
fees_as_share_total      27   102
fees_as_share_min        24    81
btc_transacted_total     52    68
btc_received_mean        40    63
btc_received_total       47    63
btc_transacted_mean      36    58
btc_transacted_median    31    49
btc_sent_mean            37    49
fees_as_share_max         7    48
btc_received_median      32    48
btc_received_max         33    48
fees_total               45    45
fees_mean                39    43
btc_transacted_min       24    41
fees_median              35    38
btc_sent_median          31    33
fees_max                 27    27
btc_sent_min             27    27
btc_transacted_max        8    24
btc_received_min          4    19
fees_min                  7     8
btc_sent_max              7     7
btc_sent_total            

In [4]:
extra = divergent['1e-8'] & ~divergent['1e-4']
print(f'addresses divergent under the one-satoshi floor only: {extra.sum()}')
rows = []
for f in BTC:
    bad = extra & ~np.isclose(C[f], P[f], rtol=1e-2, atol=1e-8)
    if bad.any():
        d = np.abs(C[f][bad] - P[f][bad])
        rows.append({'feature': f, 'addresses': int(bad.sum()), 'max |c - p|': d.max(),
                     'median |c - p|': np.median(d), 'median |p|': np.median(np.abs(P[f][bad]))})
if rows:
    print(pd.DataFrame(rows).sort_values('addresses', ascending=False).to_string(index=False))
print()
print(f"median transaction fee of the labelled addresses (released fees_median): {np.median(P['fees_median']):.5f} BTC")

addresses divergent under the one-satoshi floor only: 51
              feature  addresses  max |c - p|  median |c - p|   median |p|
   fees_as_share_mean         46     0.000066        0.000008 5.410853e-06
  fees_as_share_total         44     0.000199        0.000025 8.932700e-06
    fees_as_share_min         42     0.000094        0.000007 9.549739e-07
 fees_as_share_median         40     0.000066        0.000009 3.882189e-06
    fees_as_share_max         32     0.000070        0.000021 2.572701e-08
   btc_transacted_min         16     0.000079        0.000027 0.000000e+00
   btc_received_total         16     0.000300        0.000027 0.000000e+00
   btc_transacted_max         15     0.000027        0.000027 0.000000e+00
 btc_transacted_total         15     0.000054        0.000027 0.000000e+00
     btc_received_min         15     0.000027        0.000027 0.000000e+00
btc_transacted_median         15     0.000027        0.000027 0.000000e+00
  btc_transacted_mean         15     0.0000

## 3. Fee share

The released fee share follows a per-transaction definition, the share of each transaction's fee attributed to the address,
summarised over its transactions. The alternative reading is a single ratio of the address's total fees to its total amount transacted.

In [5]:
per_tx = C['fees_as_share_total']
total = C['btc_transacted_total']
ratio = np.divide(C['fees_total'], total, out=np.zeros(len(m)), where=total > 0)
p = P['fees_as_share_total']
for name, a in FLOORS.items():
    print(f'floor {name}: per-transaction {np.isclose(per_tx, p, rtol=1e-2, atol=a).mean():.4%}   '
          f'single ratio {np.isclose(ratio, p, rtol=1e-2, atol=a).mean():.4%}')

floor 1e-4: per-transaction 99.9898%   single ratio 7.6513%
floor 1e-8: per-transaction 99.9616%   single ratio 5.9954%


## 4. Identities among the features

On the training addresses. Integer-valued features are compared exactly, amounts with a relative tolerance of $10^{-6}$
and a floor of one satoshi.

In [6]:
train = lifetime.loc[lifetime.split == 'train', 'address']
sets = {'released':          released.set_index('address').loc[train],
        'lifetime':          lifetime.set_index('address').loc[train],
        'leakage-resistant': resistant.set_index('address').loc[train]}

IDENTITIES = {
    'total block gap = lifetime':
        lambda d: d.blocks_btwn_txs_total == d.lifetime_in_blocks,
    'lifetime = last block - first block':
        lambda d: d.lifetime_in_blocks == d.last_block_appeared_in - d.first_block_appeared_in,
    'transacted = sent + received, totals':
        lambda d: np.isclose(d.btc_transacted_total, d.btc_sent_total + d.btc_received_total, rtol=1e-6, atol=1e-8),
    'transacted = sent + received, means':
        lambda d: np.isclose(d.btc_transacted_mean, d.btc_sent_mean + d.btc_received_mean, rtol=1e-6, atol=1e-8),
    'transactions = sending + receiving':
        lambda d: d.total_txs == d.num_txs_as_sender + d['num_txs_as receiver'],
}
rows = []
for identity, test in IDENTITIES.items():
    row = {'identity': identity}
    for name, d in sets.items():
        ok = np.asarray(test(d))
        row[name] = f'{ok.mean():.3%} ({(~ok).sum()})'
    rows.append(row)
print(f'training addresses {len(train):,}; share satisfying each identity, exceptions in parentheses')
print(pd.DataFrame(rows).to_string(index=False))

training addresses 172,903; share satisfying each identity, exceptions in parentheses
                            identity     released     lifetime leakage-resistant
          total block gap = lifetime 99.971% (50) 100.000% (0)      100.000% (0)
 lifetime = last block - first block 100.000% (0) 100.000% (0)      100.000% (0)
transacted = sent + received, totals  99.998% (4) 100.000% (0)      100.000% (0)
 transacted = sent + received, means  99.997% (5) 100.000% (0)      100.000% (0)
  transactions = sending + receiving 100.000% (0)  99.997% (5)       99.998% (3)


In [7]:
rel, lt = sets['released'], sets['lifetime']
bad = rel.index[rel.blocks_btwn_txs_total != rel.lifetime_in_blocks]
excess = rel.loc[bad, 'blocks_btwn_txs_total'] - rel.loc[bad, 'lifetime_in_blocks']
print(f'released: total block gap differs from lifetime for {len(bad)} addresses')
print(f'  gap total exceeds lifetime for {(excess > 0).sum()}, by {excess.min():.0f} to {excess.max():.0f} blocks')
print(f'  our lifetime equals the released lifetime for {(lt.loc[bad, "lifetime_in_blocks"] == rel.loc[bad, "lifetime_in_blocks"]).sum()}')

released: total block gap differs from lifetime for 50 addresses
  gap total exceeds lifetime for 50, by 1 to 12 blocks
  our lifetime equals the released lifetime for 50


In [8]:
cols = ['total_txs', 'num_txs_as_sender', 'num_txs_as receiver']
zero = lambda d: d.index[d.total_txs != d.num_txs_as_sender + d['num_txs_as receiver']]
bad = zero(sets['lifetime']).union(zero(sets['leakage-resistant']))
print('addresses with a transaction of zero net stake')
print(pd.concat({name: d.reindex(bad)[cols] for name, d in sets.items()}, axis=1).to_string())

addresses with a transaction of zero net stake
                                    released                                        lifetime                                       leakage-resistant                                      
                                   total_txs num_txs_as_sender num_txs_as receiver total_txs num_txs_as_sender num_txs_as receiver         total_txs num_txs_as_sender num_txs_as receiver
address                                                                                                                                                                                   
14wXrm49HxggbdQ6RGfWY8qghGEWhLA28K     904.0             419.0               485.0       904             418.0               485.0               898             416.0               481.0
1Lsqcv4cg5zUctNi2qwNxMkrv1GeBboSUJ     989.0             474.0               515.0       989             472.0               515.0               976             465.0               509.0
1Po4J4SNyJuGnMGYJf